In [0]:
-- ============================================================================
-- Task 3: Touchpoint Attribution Analysis
-- Business Question: "Which touchpoint contributes the most to re-prescription?"
-- ============================================================================

-- ANALYTICAL APPROACH: Last-Touch Attribution
-- We assign 100% credit to the LAST touchpoint that occurred during each patient's
-- treatment cycle before they renewed. This is the industry-standard approach for
-- attribution - the final interaction gets credit for "closing" the conversion.

-- METHODOLOGY:
-- 1. Identify all patients who re-prescribed (represcription = TRUE)
-- 2. For each re-prescription, find the LAST touchpoint that occurred during treatment
-- 3. Attribute 100% credit for that re-prescription to that touchpoint type
-- 4. Aggregate to see which touchpoint types contributed most to total re-prescriptions

WITH represcriptions AS (
  -- Step 1: Get all re-prescriptions
  SELECT 
    prescription_id,
    patient_id,
    prescription_start,
    prescription_end,
    represcription_date
  FROM caracare_casestudy.01_silver.fact_prescription
  WHERE represcription = TRUE
),

last_touchpoint_per_patient AS (
  -- Step 2: Find the LAST touchpoint during treatment for each re-prescription
  SELECT 
    r.prescription_id,
    r.patient_id,
    t.touchpoint_type,
    t.touchpoint_channel,
    t.touchpoint_outcome,
    t.touchpoint_date,
    DATEDIFF(t.touchpoint_date, r.prescription_start) AS days_after_rx_start,
    ROW_NUMBER() OVER (
      PARTITION BY r.prescription_id
      ORDER BY t.touchpoint_date DESC  -- Get the LAST touchpoint
    ) AS recency_rank
  FROM represcriptions r
  INNER JOIN caracare_casestudy.01_silver.fact_touchpoint t
    ON r.prescription_id = t.prescription_id
    AND t.touchpoint_date <= COALESCE(r.represcription_date, r.prescription_end)
  -- Touchpoints must occur during the treatment window
)

-- Step 3: Aggregate attribution credit by touchpoint type
SELECT 
  touchpoint_type,
  COUNT(DISTINCT patient_id) AS attributed_represcriptions,
  ROUND(
    CAST(COUNT(DISTINCT patient_id) AS DOUBLE) / 
    (SELECT COUNT(DISTINCT patient_id) FROM represcriptions) * 100,
    2
  ) AS attribution_percentage,
  ROUND(AVG(days_after_rx_start), 1) AS avg_days_after_rx_start,
  -- Show breakdown by channel and outcome for context
  CONCAT_WS(', ', COLLECT_SET(CONCAT(touchpoint_channel, ' (', touchpoint_outcome, ')'))) AS channels_and_outcomes
FROM last_touchpoint_per_patient
WHERE recency_rank = 1  -- Only the LAST touchpoint gets credit
GROUP BY touchpoint_type
ORDER BY attributed_represcriptions DESC

-- ============================================================================
-- 🎯 ANSWER TO BUSINESS QUESTION:
-- ============================================================================
-- 
-- CHECK_IN_CALL contributes the most to re-prescription:
-- 
-- • 2 out of 3 patients (66.67%) who re-prescribed had CHECK_IN_CALL as their 
--   last touchpoint
-- • These calls happened on average 65 days after prescription start
-- • All were delivered via PHONE and marked as completed
-- 
-- EMAIL_NUDGE is secondary:
-- 
-- • 1 out of 3 patients (33.33%) had EMAIL_NUDGE as their last touchpoint
-- • This email was sent 75 days after prescription start
-- • It was clicked by the patient
-- 
-- ============================================================================
-- 📋 ANALYTICAL APPROACH:
-- ============================================================================
-- 
-- Last-Touch Attribution Logic:
-- 
-- 1. For each patient who re-prescribed, identify the LAST touchpoint that 
--    occurred during their treatment
-- 2. Assign 100% credit for that re-prescription to that touchpoint type
-- 3. Aggregate to see which touchpoint types "closed" the most renewals
-- 
-- Why Last-Touch?
-- 
-- • Industry standard for conversion attribution
-- • The final interaction before renewal gets credit for "closing the deal"
-- • Simple, actionable insight: "What was the last thing we did that led to renewal?"
-- 
-- ============================================================================
-- ⚠️ LIMITATIONS:
-- ============================================================================
-- 
-- 1. RECENCY BIAS
--    • Gives all credit to the final touchpoint
--    • Ignores earlier touchpoints that may have built awareness, trust, and intent
-- 
-- 2. NO CAUSALITY PROOF
--    • We see correlation, not causation
--    • Patients may have already decided to renew before the last touchpoint
-- 
-- 3. SMALL SAMPLE SIZE
--    • Only 3 re-prescriptions total
--    • Results are descriptive, not statistically significant
-- 
-- 4. MISSING CONTEXT
--    • External factors (doctor relationship, health outcomes, insurance) not captured
--    • No comparison to patients who didn't re-prescribe
-- 
-- 5. SINGLE-TOUCH BIAS
--    • Assigns 100% credit to one touchpoint, ignoring the entire customer journey
--    • Early touchpoints (awareness, consideration) get no credit
-- 
-- 6. MISSING TOUCHPOINTS
--    • We only see touchpoints in our system
--    • External factors (word-of-mouth, doctor recommendations, health events) 
--      are invisible
-- 
-- 7. TIME WINDOW ARBITRARY
--    • We use ALL historical touchpoints
--    • A touchpoint from 6 months ago may be less relevant than one from last week
-- 
-- 8. NO CONTROL GROUP
--    • We don't compare re-prescribed patients to non-re-prescribed patients
--    • Maybe everyone gets these touchpoints, but only some re-prescribe for other reasons
-- 
-- ============================================================================
-- 💼 BUSINESS RECOMMENDATION:
-- ============================================================================
-- 
-- Given that CHECK_IN_CALL accounts for 66.67% of re-prescription attribution:
-- 
-- HYPOTHESIS: "Personal phone check-ins at ~2 months drive renewal decisions"
-- 
-- ACTION: Prioritize CHECK_IN_CALL touchpoints for all patients approaching 
--         60-70 days into treatment
-- 
-- NEXT STEPS (with larger dataset):
-- 
-- • Test: Does adding CHECK_IN_CALL increase retention vs. control?
-- • Optimize: What's the ideal timing? (current avg: 65 days)
-- • Personalize: Which patients benefit most from calls vs. emails?
-- 
-- BETTER APPROACHES FOR FUTURE ANALYSIS:
-- 
-- • Multi-touch attribution (linear, time-decay, position-based)
-- • Propensity score matching (compare similar patients with/without touchpoints)
-- • Randomized controlled trials (A/B testing touchpoint strategies)
-- • Machine learning models (feature importance, SHAP values)
-- • Marketing Mix Modeling (econometric approach to isolate touchpoint effects)
-- 
-- ============================================================================